# 1. Imports

In [35]:
import pandas as pd

from langchain_ollama import ChatOllama
from langchain_experimental.agents.agent_toolkits import (
    create_pandas_dataframe_agent,
)

from pydantic import BaseModel, Field
from typing import Any


# 2. Load the CSV

In [ ]:
DATA_PATH = "../sample_data/physical_exam_study.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,gender,intensity,pain_score
0,m,4.6,6
1,f,6.0,5
2,f,5.5,3
3,m,4.1,6
4,m,3.6,5


## Print dataframe info

In [5]:
print("Shape:", df.shape)
print()
print("Columns:")
print(df.dtypes)
print()
print("Summary statistics:", df.describe(include="all"))
print("Genders:", df["gender"].value_counts())
print("Missing values:")
print(df.isna().sum())

Shape: (2000, 3)

Columns:
gender         object
intensity     float64
pain_score      int64
dtype: object

Summary statistics:        gender    intensity   pain_score
count    2000  2000.000000  2000.000000
unique      2          NaN          NaN
top         m          NaN          NaN
freq     1000          NaN          NaN
mean      NaN     4.165700     4.153500
std       NaN     1.097413     2.984022
min       NaN     0.200000     0.000000
25%       NaN     3.400000     1.000000
50%       NaN     4.200000     4.000000
75%       NaN     4.900000     7.000000
max       NaN     8.300000    10.000000
Genders: gender
m    1000
f    1000
Name: count, dtype: int64
Missing values:
gender        0
intensity     0
pain_score    0
dtype: int64


# Calculate stats with Pandas

## Mean / standard deviation for men

In [6]:
male_intensity = df.loc[df["gender"] == "m", "intensity"]

print("Mean:", male_intensity.mean())
print("Std:", male_intensity.std())

Mean: 4.2542
Std: 1.0731607680278077


## Pain min/max for women

In [7]:
female_pain = df.loc[df["gender"] == "f", "pain_score"]

print("Min:", female_pain.min())
print("Max:", female_pain.max())

Min: 0
Max: 7


## Correlation

In [8]:
overall_corr = df["intensity"].corr(df["pain_score"])

male_corr = (
    df.loc[df["gender"] == "m", "intensity"]
    .corr(df.loc[df["gender"] == "m", "pain_score"])
)

female_corr = (
    df.loc[df["gender"] == "f", "intensity"]
    .corr(df.loc[df["gender"] == "f", "pain_score"])
)

print("Overall:", overall_corr)
print("Men:", male_corr)
print("Women:", female_corr)

Overall: 0.5032174582559054
Men: 0.8390746332687103
Women: 0.8163610031517332


# 4. Configure Ollama

In [54]:
from pydantic import BaseModel, Field


MODEL_NAME = "gemma4:e2b"

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
)


ScalarValue = int | float | str | bool | None


class DataAnalysisAnswer(BaseModel):
    summary: str
    result: dict[str, ScalarValue]


class AnalysisSummary(BaseModel):
    summary: str

summary_llm = llm.with_structured_output(
    AnalysisSummary,
    method="json_schema",
)

# 5. Build the baseline Pandas agent

In [55]:
AGENT_PREFIX = """
You are a data analysis agent working with a pandas DataFrame named `df`.

Always use the Python tool for calculations.

Perform the required calculations and make the final successful
Python tool call print exactly one dictionary containing all requested
values with clear keys.

Do not print intermediate calculation results unless needed for debugging.

All dictionary values must be standard Python values.
Convert pandas or NumPy numeric values using float() or int().

Example:

result = {
    "mean": float(mean_value),
    "std": float(std_value),
}
print(result)
"""


agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    agent_type="tool-calling",
    prefix=AGENT_PREFIX,
    verbose=True,
    allow_dangerous_code=True,
    return_intermediate_steps=True,
    max_iterations=10,
)

# 6. Retrieval queries

## 1. simple test

In [51]:
result = agent.invoke(
    "What are the minimum and maximum values "
    "of the pain score for women?",
    verbose=True,
)

print(result["output"])
print(result.keys())



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "print(df[df['gender'] == 'f']['pain_score'].min())\nprint(df[df['gender'] == 'f']['pain_score'].max())"}`


0
7

Invoking: `python_repl_ast` with `{'query': 'min_pain_score = df[df[\'gender\'] == \'f\'][\'pain_score\'].min()\nmax_pain_score = df[df[\'gender\'] == \'f\'][\'pain_score\'].max()\nprint({"min_pain_score": float(min_pain_score), "max_pain_score": float(max_pain_score)})'}`


{'min_pain_score': 0.0, 'max_pain_score': 7.0}
{
    "min_pain_score": 0.0,
    "max_pain_score": 7.0
}

> Finished chain.
{
    "min_pain_score": 0.0,
    "max_pain_score": 7.0
}
dict_keys(['input', 'output', 'intermediate_steps'])


## 2. Setup ask-answer functions

In [56]:
import ast
import re


def extract_tool_result(agent_result: dict) -> dict:
    for _, observation in reversed(agent_result["intermediate_steps"]):
        text = str(observation).strip()

        # Fallback for NumPy scalar representations such as:
        # np.float64(4.2542) -> 4.2542
        text = re.sub(
            r"np\.(?:float|int)\d+\(([^()]+)\)",
            r"\1",
            text,
        )

        try:
            result = ast.literal_eval(text)

            if isinstance(result, dict):
                return result

        except (ValueError, SyntaxError):
            pass

    raise ValueError(
        "No structured dictionary was returned by the dataframe tool."
    )


def ask(question: str) -> DataAnalysisAnswer:
    print(f"\n{'=' * 80}")
    print(f"Question: {question}")
    print("=" * 80)

    # Run dataframe analysis
    agent_result = agent.invoke(question)

    # Exact values from Python execution
    computed_result = extract_tool_result(agent_result)

    print("\nComputed result:")
    print(computed_result)

    # LLM generates explanation only
    summary_result = summary_llm.invoke(
        f"""
    Question:
    {question}

    Computed result:
    {computed_result}

    Explain the result briefly.
    Use only the supplied values.
    Do not claim statistical significance unless a statistical
    significance test was explicitly computed.
    """
    )

    # Combine LLM summary with untouched Python result
    final_result = DataAnalysisAnswer(
        summary=summary_result.summary,
        result=computed_result,
    )

    print("\nFinal answer:")
    print(final_result)

    return final_result

In [57]:
result_1 = ask(
    "What is the mean and standard deviation "
    "of the workout intensity for men?"
)

result_2 = ask(
    "What are the minimum and maximum values "
    "of the pain score for women?"
)

result_3 = ask(
    "Is there a correlation between workout intensity "
    "and perceived pain? Calculate the Pearson correlation "
    "for the complete dataset and separately for men and women. "
    "Does the correlation differ between men and women?"
)


Question: What is the mean and standard deviation of the workout intensity for men?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'mean_intensity_men = df[df[\'gender\'] == \'m\'][\'intensity\'].mean()\nstd_intensity_men = df[df[\'gender\'] == \'m\'][\'intensity\'].std()\nprint({"mean": float(mean_intensity_men), "std": float(std_intensity_men)})'}`


{'mean': 4.2542, 'std': 1.0731607680278077}
{
    "mean": 4.2542,
    "std": 1.0731607680278077
}

> Finished chain.

Computed result:
{'mean': 4.2542, 'std': 1.0731607680278077}

Final answer:
summary='The mean workout intensity for men is 4.2542, and the standard deviation is 1.0731607680278077.' result={'mean': 4.2542, 'std': 1.0731607680278077}

Question: What are the minimum and maximum values of the pain score for women?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'result = df[df[\'gender\'] == \'f\']\nmin_pain_score = result[\'pain_score\'].min()\nmax_